# CDC Seasonal Vaccine Prediction Model

The Goal of this file is to try and predict the vaccination rate (as a %) of a given county using a machine learning model. Using the model, Counties with low vaccination rates can be identified ahead of time. Once it is known which counties have the lowest vaccination rates, greater outreach efforts and media prescence can be implemented in an attempt to boost the actual vaccination rate compared to the the rate forecasted by the model.

The model uses data provided from the CDC. Follow the steps to generate and run the model.

Visit the CDC Website and preview the dataset for "Influenza Vaccination Coverage for All Ages (6+ Months)" here:

https://data.cdc.gov/Flu-Vaccinations/Influenza-Vaccination-Coverage-for-All-Ages-6-Mont/vh55-3he6/about_data

Once there, Select "Export" on the right side of the page and download the dataset as a CSV file. 
Then copy the file path from the document and paste it into the input below.

In [1]:
from pathlib import Path

#User input filepath
file_path = input("Copy and Paste the file path here: ")

#Remove accidental quotes if the user pasted "C:\path\to\file.csv"
file_path = file_path.strip().strip('"').strip("'")

#Normalize or replace extra slashes if needed
file_path = file_path.replace("\\\\", "\\")

#Convert to a Path object and resolve issues like '~' home shortcuts
file_path = Path(file_path).expanduser().resolve()

#Check if the path actually exists and is a file
if file_path.is_file():
    #Print success if it is
    print(f"Success! Found file: {file_path.name}")
else:
    #print Error if it isn't
   print(f"Error: The path '{file_path}' does not point to a valid file.")

Copy and Paste the file path here:  "C:\Users\yousu\Downloads\Influenza_Vaccination_Coverage_for_All_Ages_(6+_Months)_20261020.csv"


Success! Found file: Influenza_Vaccination_Coverage_for_All_Ages_(6+_Months)_20261020.csv


In [3]:
#Preprocessing
#Inport the CSV into Python as a dataframe
df = pd.read_csv(file_path)

#We are only interested in the Seasonal Vaccine so this filters the data to only the Seasonal Influenza Vaccine
df = df[df['Vaccine'] == 'Seasonal Influenza']

#We are only interested in counties so this statement filters out states/etc and leaves only counties
df = df[df['Geography Type'] == 'Counties']

#After filtering the above are 2 dimensions that are left in the Dataframe at this point for Dimension Type; "Age" and ">=18 Years". We want Age
df = df[df['Dimension Type'] == 'Age']

#There are counties that have the same name but are in different states. Concatenating County Name with FIPS yields a unique County Categorical field
df['County'] = df['Geography'].astype(str)+df['FIPS'].astype(str)

#Here there are a few features that have been exluded due to the fact that they are all the same value; Month(For counties, all months are '1'), Vaccine(we are testing only the Seasonal Influenza Vaccine), Dimension (The dimensions are all the same for counties (Age6+), and Sample Size(NA for Counties) exist but offer nothing to the model
df = df[['Season/Survey Year','County','Estimate (%)']]

#Need to cast "Season/Survey Year" as an integer to make it easier to filter later on
df['Season/Survey Year'] = df['Season/Survey Year'].astype(int)

#Convert Estimate to a number data type (it is object currently)
df['Estimate (%)'] = df['Estimate (%)'].astype(float)

# Convert County to a category type for our model later
df['County'] = df['County'].astype('category')

# Sort by year to ensure correct temporal ordering
df = df.sort_values(by='Season/Survey Year').reset_index(drop=True)

In [4]:
#Need to get the latest year +1 as our predict
last_year = int(df['Season/Survey Year'].max())
future_year = last_year + 1

In [5]:
#Feature Engineering

# To capture true time-series dependency, we track what the percentage was the year prior and add it to our Dataframe as a new field
df['Estimate (%) Lag'] = df.groupby('County', observed = False)['Estimate (%)'].shift(1)

# Drop the first year since it won't have a lag value
df = df.dropna().reset_index(drop=True)

In [6]:
#Supervised Training

# We train on years up to the last, and test on the last year
train_set= df[df['Season/Survey Year'] < last_year]
test_set= df[df['Season/Survey Year'] >= last_year]

#Create Train set
y_train = train_set['Estimate (%)']
y_train = y_train.reset_index(drop=True)

X_train = train_set.drop(['Estimate (%)'], axis = 1)
X_train = X_train.reset_index(drop=True)

#Create Test Set
y_test = test_set['Estimate (%)']
y_test = y_test.reset_index(drop=True)

X_test = test_set.drop(['Estimate (%)'], axis = 1)
X_test = X_test.reset_index(drop=True)

In [7]:
#Initialize XGB Regressor
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=300,         # Number of sequential decision trees to build in the ensemble
    learning_rate=0.05,       # Change ech tree contributes to the ensemble
    max_depth=6,              # Maximum number of sequential splits from the root node
    enable_categorical=True,  # Required for categorical feature support
    tree_method='hist',       # Crucial for high-cardinality processing
    random_state=42
)

In [8]:
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [9]:
from sklearn.metrics import mean_squared_error

preds = xgb_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"Test RMSE for Year 2025: {rmse:.4f}")

Test RMSE for Year 2025: 3.7814


In [10]:
# Look at feature importances
importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
print("\nFeature Importances:")
print(importances)


Feature Importances:
Season/Survey Year    0.196773
County                0.207003
Estimate (%) Lag      0.596224
dtype: float32


In [11]:
#Construct a Future Dataframe for the year after the test year data
future_df = pd.DataFrame({
    'Season/Survey Year': [future_year]*len(test_set['County']),
    'County':test_set['County'],
    'Estimate (%) Lag': test_set.sort_values('County')['Estimate (%)'].values
})

#Predict from the Future Dataframe
future_preds = xgb_model.predict(future_df)

#Add it to our future_df
future_df['Predicted Estimate (%)'] = future_preds

In [24]:
#Predict the 3 counties with the lowest vaccination rate predicteed by the model in the future year
low_vax = future_df.nsmallest(3, 'Predicted Estimate (%)')
low_vax['FIPS'] = low_vax['County'].str[-5:]
low_vax['County'] = low_vax['County'].str[:-5]
print('The 3 counties with lowest vaccination rate predicted in the year ' +str(future_year)+ ' are:')
print(low_vax[['County', 'FIPS', 'Predicted Estimate (%)']].to_string(index=False))

The 3 counties with lowest vaccination rate predicted in the year 2023 are:
  County  FIPS  Predicted Estimate (%)
Hudspeth 48229               29.101374
 Stewart 13259               29.267345
  Dawson 48115               29.407732
